# Week 5 exercise — RAG over the knowledge base

**RAG** = Retrieval-Augmented Generation: instead of stuffing everything into the prompt,
store documents as vectors, **retrieve** only the few relevant ones for a question, and feed
those to the model. Here: index the Insurellm knowledge base in a Chroma vector DB (local
embeddings), then answer questions grounded in the retrieved docs.

In [1]:
from pathlib import Path
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
frontier = OpenAI()

# find the course knowledge base by walking up from the working dir (robust to where this runs)
KB = next(p / "week5/knowledge-base" for p in [Path.cwd(), *Path.cwd().parents]
          if (p / "week5/knowledge-base").exists())
print("knowledge base:", KB)

knowledge base: C:\Users\Nicholas Dean\projects\llm_engineering\week5\knowledge-base


In [2]:
# Load every doc and index it. Chroma's default embedder runs locally (MiniLM, no API cost).
docs, ids, metas = [], [], []
for f in sorted(KB.rglob("*.md")):
    docs.append(f.read_text(encoding="utf-8"))           # the document text
    ids.append(str(f.relative_to(KB)))                   # a unique id (its relative path)
    metas.append({"category": f.parent.name})            # company / products / employees / contracts

collection = chromadb.Client().create_collection("insurellm")
collection.add(documents=docs, ids=ids, metadatas=metas)  # embeds + stores all docs
print(f"indexed {collection.count()} documents")

indexed 76 documents


In [3]:
def rag_answer(question, k=4):
    hits = collection.query(query_texts=[question], n_results=k)   # retrieve top-k relevant docs
    context = "\n\n---\n\n".join(hits["documents"][0])             # stitch them into context
    messages = [
        {"role": "system", "content": "Answer using ONLY the context. If it's not there, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    answer = frontier.chat.completions.create(model="gpt-4o-mini", messages=messages).choices[0].message.content
    return answer, ids_from(hits)

def ids_from(hits):
    return hits["ids"][0]

for q in ["Who is the CEO of Insurellm?", "What products does Insurellm offer?"]:
    ans, sources = rag_answer(q)
    print(f"Q: {q}\nA: {ans}\n   sources: {sources}\n")

Q: Who is the CEO of Insurellm?
A: The CEO of Insurellm is Avery Lancaster.
   sources: ['company\\about.md', 'company\\overview.md', 'employees\\Avery Lancaster.md', 'company\\careers.md']



Q: What products does Insurellm offer?
A: Insurellm offers 8 insurance software products across multiple insurance lines:

### Core Insurance Portals
- **Carllm** - Auto insurance platform for insurers
- **Homellm** - Home insurance platform for insurers
- **Lifellm** - Life insurance platform with AI-powered underwriting
- **Healthllm** - Comprehensive health insurance platform
- **Bizllm** - Commercial insurance platform for business coverage

### Marketplace & Infrastructure
- **Markellm** - Marketplace connecting consumers with insurance providers (original flagship product)
- **Claimllm** - AI-powered claims processing platform across all insurance lines
- **Rellm** - Enterprise platform for the reinsurance sector
   sources: ['company\\about.md', 'company\\overview.md', 'contracts\\Contract with EverGuard Insurance for Rellm - AI-Powered Enterprise Reinsurance Solution.md', 'contracts\\Contract with National Claims Network for Claimllm.md']

